## Templates as functions

In [1]:
from langchain_openai import ChatOpenAI
import getpass

In [2]:
OPENAI_API_KEY = getpass.getpass('Enter your OPENAI_API_KEY')

Enter your OPENAI_API_KEY ········


In [5]:
llm = ChatOpenAI(
    base_url="https://openrouter.ai/api/v1",
    openai_api_key=OPENAI_API_KEY,
    model_name="gpt-5-nano"
)

In [7]:
input_prompt = """Classify the following numbers as Abra, Kadabra or Abra Kadabra:
3, 4, 5, 7, 8, 10, 11, 13, 35
Examples:
6 // not divisible by 5, not divisible by 7 // None
15 // divisible by 5, not divisible by 7 // Abra
12 // not divisible by 5, not divisible by 7 // None
21 // not divisible by 5, divisible by 7 // Kadabra
70 // divisible by 5, divisible by 7 // Abra Kadabra
"""

In [8]:
response = llm.invoke(input_prompt)

In [9]:
print(response.content)

- 3: None
- 4: None
- 5: Abra
- 7: Kadabra
- 8: None
- 10: Abra
- 11: None
- 13: None
- 35: Abra Kadabra


The output is accurate, but the implementation isn’t ideal because the examples are hardcoded in the prompt.

LangChain lets you **separate the training examples from the prompt template** and inject them later, as shown in the following listing.

In [10]:
from langchain_core.prompts.few_shot import FewShotPromptTemplate
from langchain_core.prompts.prompt import PromptTemplate

In [12]:
examples = [
    {
        "number": 6,
        "reasoning": "not divisible by 5 nor by 7",
        "result": "None"
    },
    {
        "number": 15,
        "reasoning": "divisible by 5 but not by 7",
        "result": "Abra"
    },
    {
        "number": 12,
        "reasoning": "not divisible by 5 nor by 7",
        "result": "None"
    },
    {
        "number": 21,
        "reasoning": "divisible by 7 but not by 5",
        "result": "Kadabra"
    },
    {
        "number": 70,
        "reasoning": "divisible by 5 and by 7",
        "result": "Abra Kadabra"
    }
]

In [13]:
example_prompt = PromptTemplate(
    input_variables=["number", "reasoning", "result"],
    template="{number} \\ {reasoning} \\ {result}"
)

In [14]:
few_shot_prompt = FewShotPromptTemplate(
    examples=examples,
    example_prompt=example_prompt,
    suffix="Classify the following numbers as Abra, Kadabra or Abra Kadabra: {comma_delimited_input_numbers}",
    input_variables=["comma_delimited_input_numbers"]
)

In [16]:
prompt_input = few_shot_prompt.format(comma_delimited_input_numbers="3, 4, 5, 7, 8, 10, 11, 13, 35.")

In [17]:
response = llm.invoke(prompt_input)

In [18]:
print(response.content)

Rule: Abra = divisible by 5 only; Kadabra = divisible by 7 only; Abra Kadabra = divisible by both; None = divisible by neither.

- 3: None
- 4: None
- 5: Abra
- 7: Kadabra
- 8: None
- 10: Abra
- 11: None
- 13: None
- 35: Abra Kadabra
